# Step 1 of RAG: Multiformat Document Loading & Advanced Chunking Strategies

**Environment:** Google Colab / Local Python 3.9+  
**Domain:** Enterprise AI Governance, Security Protocols & Retrieval Architecture

---

### What You Will Learn
1. **The RAG Preprocessing Foundation:** Why document loading and chunking dictates 80% of RAG retrieval quality.
2. **The Chunking Tradeoff:** Navigating *Context Fragmentation* (chunks too small) vs. *Context Dilution* (chunks too large).
3. **Multiformat Loaders:** Loading and parsing Plain Text (`.txt`), Markdown (`.md`), PDF (`.pdf`), and Structured JSON (`.json`).
4. **6 Core Chunking Strategies:**
   - Strategy 1: Fixed-Character Sliding Window (From scratch)
   - Strategy 2: Sentence-Boundary Aware Chunking (Regex with abbreviation protection)
   - Strategy 3: LangChain `CharacterTextSplitter`
   - Strategy 4: LangChain `RecursiveCharacterTextSplitter`
   - Strategy 5: Token-Based Chunking (`tiktoken` for LLM context windows)
   - Strategy 6: **Semantic Chunking** (Adaptive splitting based on embedding distance jumps)
5. **Comparative Statistical Analytics:** Distribution histograms, boxplots, and sentence-boundary integrity scoring.
6. **Production Ingestion Pipeline:** Building a modular, metadata-preserving `RAGIngestionPipeline` class.
7. **Hands-On Exercises:** Markdown header splitting and token budgeting for LLM generation.

In [ ]:
# Install dependencies for document loading and chunking
!pip install -q langchain-text-splitters tiktoken pypdf fpdf2 pandas matplotlib seaborn

## 1. The RAG Ingestion Pipeline & The Chunking Tradeoff

Retrieval-Augmented Generation (RAG) follows four fundamental stages:
1. **Document Ingestion & Chunking** (This Notebook)
2. **Embedding Generation** (Notebook 03)
3. **Vector Database Indexing** (Notebooks 04 & 05)
4. **Context Retrieval & Generation**

```
[Raw Document] -> [Format Parser] -> [Chunking Strategy] -> [Metadata Enrichment] -> [Vector DB]
```

### The Chunking Dilemma
- **Chunk Size Too Small (< 100 tokens):** Sentences are severed from surrounding context. The model retrieves isolated fragments without sufficient detail to formulate a coherent answer.
- **Chunk Size Too Large (> 1000 tokens):** The embedding averages across multiple distinct topics (*context dilution*). Lowers semantic precision and consumes excessive LLM context tokens.

## 2. Multiformat Document Ingestion

Let us prepare our technical document corpus on disk to test loaders for `.txt`, `.md`, `.pdf`, and `.json`.

In [ ]:
import os

os.makedirs("docs_vault", exist_ok=True)

# Technical enterprise document sample
governance_manual = """# Enterprise AI Governance & Safety Protocols
Version: 3.2 | Classification: Confidential | Author: AI Platform Engineering Team

## 1. Executive Summary & Architectural Scope
Modern enterprise Generative AI systems rely on hybrid retrieval-augmented generation (RAG) pipelines to anchor Large Language Models (LLMs) to verifiable enterprise ground truth. By decoupling parameterized internal knowledge from dynamic external retrieval, organizations mitigate hallucinations, enforce strict data access controls, and provide traceable citations for downstream decision-making.

## 2. Ingestion & Document Preprocessing Standards
Document ingestion represents the critical first frontier in vector pipeline engineering. Ingested artifacts originate from heterogeneous sources:
- Unstructured text transcripts and RFC specification documents.
- Semistructured Markdown technical manuals and knowledge base articles.
- High-fidelity PDF technical manuals containing hierarchical headers and tables.
- Machine-generated JSON audit logs and structured schema definitions.

When parsing documents, ingestion pipelines must maintain clean provenance metadata including document ID, source URI, page numbers, creation timestamp, and cryptographic checksums.

## 3. Vector Embeddings & Metric Spaces
Vector embeddings map discrete textual tokens into dense, continuous vector spaces where geometric proximity reflects semantic similarity. Unlike classical sparse one-hot vectors—which suffer from the curse of dimensionality and complete orthogonality—dense embeddings capture synonymy, topical relatedness, and conceptual analogies.

Engineers must evaluate representation dimensionality (e.g., 384 vs 768 dimensions), inference latency, and memory footprint when deploying embedding models.
"""

# Save Plain Text file
txt_path = "docs_vault/ai_governance.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(governance_manual)

print(f"Created sample text file at: {txt_path} ({len(governance_manual)} characters)")

### 2.1 Generating and Loading a PDF Document with `fpdf2` and `pypdf`

In [ ]:
from fpdf import FPDF
from pypdf import PdfReader

pdf_path = "docs_vault/ai_governance.pdf"

# 1. Generate clean PDF
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=11)
for line in governance_manual.split("
"):
    # Strip non-latin characters for simple pdf font compatibility
    clean_line = line.encode("latin-1", "replace").decode("latin-1")
    pdf.multi_cell(0, 6, clean_line)
pdf.output(pdf_path)

# 2. Extract text with PyPDF
reader = PdfReader(pdf_path)
extracted_pdf_pages = []
for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()
    extracted_pdf_pages.append({"page": page_num, "text": page_text})

print(f"Successfully loaded PDF: {pdf_path}")
print(f"Total Pages: {len(extracted_pdf_pages)}")
print(f"Extracted Character Count: {len(extracted_pdf_pages[0]['text'])}")

## 3. Exploring 6 Distinct Chunking Strategies

Now we take our extracted plain text and systematically dissect 6 chunking strategies.

### Strategy 1: Fixed-Character Chunking with Overlap (From Scratch)
A sliding window moves across the string with step size `stride = chunk_size - overlap`.

In [ ]:
def fixed_character_chunker(text, chunk_size=350, overlap=50):
    """Custom sliding window fixed-character chunker."""
    if overlap >= chunk_size:
        raise ValueError("Overlap must be strictly smaller than chunk_size!")
        
    chunks = []
    start = 0
    stride = chunk_size - overlap
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append({
            "chunk_id": len(chunks),
            "start": start,
            "end": min(end, len(text)),
            "text": chunk
        })
        start += stride
    return chunks

fixed_chunks = fixed_character_chunker(governance_manual, chunk_size=350, overlap=50)
print(f"Fixed Chunks Generated: {len(fixed_chunks)}")
print("\nFirst Chunk Preview:")
print(f"[{fixed_chunks[0]['text']}]")
print("\nNotice how the boundary ends:")
print(f"Boundary: ...{fixed_chunks[0]['text'][-25:]}")

Notice the major drawback of fixed-character chunking: it indiscriminately cuts words in half (e.g. `docu-` and `ment`) and bisects sentences.

### Strategy 2: Sentence-Boundary Aware Chunking (Regex)
Preserves complete sentences by splitting on punctuation (`.`, `!`, `?`) while protecting common abbreviations (`e.g.`, `i.e.`, `vs.`, `Dr.`).

In [ ]:
import re

def sentence_aware_chunker(text, max_sentences_per_chunk=3):
    """Splits by sentence boundaries while protecting common abbreviations."""
    # Protect common technical abbreviations
    text_protected = text
    abbreviations = [("e.g.", "EG_PLACEHOLDER"), ("i.e.", "IE_PLACEHOLDER"), ("vs.", "VS_PLACEHOLDER")]
    for abbr, placeholder in abbreviations:
        text_protected = text_protected.replace(abbr, placeholder)
        
    # Split on sentence terminals followed by whitespace
    raw_sentences = re.split(r'(?<=[.!?])\s+', text_protected)
    
    # Restore abbreviations
    sentences = []
    for s in raw_sentences:
        clean_s = s
        for abbr, placeholder in abbreviations:
            clean_s = clean_s.replace(placeholder, abbr)
        if clean_s.strip():
            sentences.append(clean_s.strip())
            
    # Group sentences into chunks
    chunks = []
    for i in range(0, len(sentences), max_sentences_per_chunk):
        grouped = " ".join(sentences[i:i + max_sentences_per_chunk])
        chunks.append(grouped)
    return chunks

sentence_chunks = sentence_aware_chunker(governance_manual, max_sentences_per_chunk=2)
print(f"Sentence-Aware Chunks Generated: {len(sentence_chunks)}")
print(f"Sample Chunk 1: {sentence_chunks[0][:150]}...")

### Strategy 3: LangChain `CharacterTextSplitter`
Splits only on specific delimiters (such as double newlines `\n\n` representing paragraphs).

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=400,
    chunk_overlap=50
)

char_chunks = char_splitter.split_text(governance_manual)
print(f"CharacterTextSplitter Chunks: {len(char_chunks)}")
for i, c in enumerate(char_chunks[:2]):
    print(f"\nChunk #{i+1} (Length: {len(c)} chars):\n{c[:100]}...")

### Strategy 4: LangChain `RecursiveCharacterTextSplitter`
The recommended industry standard for unstructured prose. It attempts to split on `["\n\n", "\n", " ", ""]` in descending order, preserving paragraphs and sentences before resorting to character-level cuts.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(governance_manual)
print(f"RecursiveCharacterTextSplitter Chunks: {len(recursive_chunks)}")
for i, c in enumerate(recursive_chunks[:2]):
    print(f"\nRecursive Chunk #{i+1} (Length: {len(c)} chars):\n{c[:100]}...")

### Strategy 5: Token-Based Chunking (`tiktoken`)
Large Language Models have maximum context budgets measured in **tokens** (subwords), not characters. In English, 1 token is roughly $0.75$ words (or $\approx 4$ characters). Splitting directly on tokens guarantees you never exceed model context windows.

In [ ]:
import tiktoken

# Use cl100k_base tokenizer (used by GPT-4 / text-embedding-ada-002)
tokenizer = tiktoken.get_encoding("cl100k_base")

def token_chunker(text, max_tokens=70, overlap_tokens=10):
    """Splits text based on exact token count boundaries."""
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    stride = max_tokens - overlap_tokens
    
    while start < len(tokens):
        chunk_tokens = tokens[start:start + max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append({
            "token_count": len(chunk_tokens),
            "text": chunk_text
        })
        start += stride
    return chunks

token_chunks = token_chunker(governance_manual, max_tokens=70, overlap_tokens=10)
print(f"Token-Based Chunks Generated: {len(token_chunks)}")
print(f"Sample Token Chunk 1 (Tokens: {token_chunks[0]['token_count']}):\n{token_chunks[0]['text'][:100]}...")

### Strategy 6: Semantic Chunking (Advanced)
Semantic chunking calculates cosine distances between embeddings of adjacent sentences. When the distance exceeds an adaptive threshold (indicating a topical transition), a new chunk boundary is created!

In [ ]:
def simple_semantic_chunker(text, distance_threshold=0.35):
    """
    Adaptive semantic chunking based on sentence embedding distance shifts.
    """
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np
    
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    # Split into individual sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) <= 1:
        return sentences
        
    embeddings = model.encode(sentences, normalize_embeddings=True)
    
    # Calculate distance between sentence i and sentence i+1
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(len(sentences) - 1):
        cos_sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        dist = 1.0 - cos_sim
        
        # If topical jump is high, start a new chunk
        if dist > distance_threshold:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i+1]]
        else:
            current_chunk.append(sentences[i+1])
            
    if current_chunk:
        chunks.append(" ".join(current_chunk))
        
    return chunks

try:
    semantic_chunks = simple_semantic_chunker(governance_manual, distance_threshold=0.35)
    print(f"Semantic Chunks Generated: {len(semantic_chunks)}")
    print(f"Sample Semantic Chunk 1: {semantic_chunks[0][:120]}...")
except Exception as e:
    print(f"Semantic chunker skipped: {e}")
    semantic_chunks = recursive_chunks

## 4. Comprehensive Comparative Analytics & Visualizations

Let us benchmark the resulting chunk length distributions and verify **sentence boundary integrity** (checking whether chunks end cleanly on punctuation marks).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

strategies = {
    "Fixed-Size": [c["text"] for c in fixed_chunks],
    "Sentence-Aware": sentence_chunks,
    "CharacterSplitter": char_chunks,
    "RecursiveCharacter": recursive_chunks,
    "Token-Based": [c["text"] for c in token_chunks],
    "Semantic": semantic_chunks
}

metrics = []
for name, chunk_list in strategies.items():
    lengths = [len(c) for c in chunk_list]
    # Check if last non-whitespace character is a terminal punctuation (. ? ! 
)
    boundary_intact = sum(1 for c in chunk_list if c.strip()[-1:] in [".", "!", "?", "#", ":"])
    integrity_pct = (boundary_intact / len(chunk_list)) * 100 if chunk_list else 0
    
    metrics.append({
        "Strategy": name,
        "Total Chunks": len(chunk_list),
        "Mean Chars": round(float(np.mean(lengths)), 1),
        "Std Dev": round(float(np.std(lengths)), 1),
        "Min Chars": np.min(lengths),
        "Max Chars": np.max(lengths),
        "Sentence Boundary Integrity (%)": round(integrity_pct, 1)
    })

df_comparison = pd.DataFrame(metrics)
display(df_comparison)

### 4.1 Visualizing Chunk Count & Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total Chunks Produced
axes[0].bar(df_comparison["Strategy"], df_comparison["Total Chunks"], color="#3b82f6", edgecolor="black")
axes[0].set_title("Total Chunks Produced by Strategy", fontsize=12, weight="bold")
axes[0].set_ylabel("Chunk Count")
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Plot 2: Sentence Boundary Integrity
axes[1].bar(df_comparison["Strategy"], df_comparison["Sentence Boundary Integrity (%)"], color="#10b981", edgecolor="black")
axes[1].set_title("Sentence Boundary Preservation (%)", fontsize=12, weight="bold")
axes[1].set_ylabel("Integrity Percentage (%)")
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Production Ingestion Pipeline Module

Here we build a production-ready `RAGIngestionPipeline` class that parses documents, chunks them, computes token counts, and attaches provenance metadata ready for ChromaDB.

In [ ]:
from typing import List, Dict, Any
import uuid

class RAGIngestionPipeline:
    """Production ingestion pipeline generating enriched chunk records."""
    
    def __init__(self, chunk_size=350, chunk_overlap=50):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        self.encoder = tiktoken.get_encoding("cl100k_base")

    def process_file(self, file_path: str, doc_category: str) -> List[Dict[str, Any]]:
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        text_chunks = self.splitter.split_text(raw_text)
        enriched_chunks = []
        
        for idx, chunk in enumerate(text_chunks):
            token_count = len(self.encoder.encode(chunk))
            chunk_record = {
                "id": f"chunk-{uuid.uuid4().hex[:8]}",
                "document": chunk,
                "metadata": {
                    "source_file": os.path.basename(file_path),
                    "chunk_index": idx,
                    "category": doc_category,
                    "char_count": len(chunk),
                    "token_count": token_count
                }
            }
            enriched_chunks.append(chunk_record)
            
        return enriched_chunks

# Test pipeline on our governance text file
pipeline = RAGIngestionPipeline(chunk_size=300, chunk_overlap=40)
records = pipeline.process_file("docs_vault/ai_governance.txt", doc_category="Governance")

print(f"Pipeline processed {len(records)} chunks ready for vector indexing.")
print("Sample Enriched Record:")
print(f"ID: {records[0]['id']}")
print(f"Metadata: {records[0]['metadata']}")
print(f"Text Snippet: {records[0]['document'][:80]}...")

## 6. Practice Exercises (Solved)

### Exercise 1: Overlap vs. Redundancy Analysis
Test `RecursiveCharacterTextSplitter` with `chunk_overlap = 0`, `50`, and `150` on the same text. Measure the total character footprint.

In [ ]:
for ovlp in [0, 50, 150]:
    s = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=ovlp)
    splits = s.split_text(governance_manual)
    total_chars = sum(len(c) for c in splits)
    print(f"Overlap = {ovlp:<3} -> Chunks: {len(splits):<2} | Total Chars: {total_chars} (+{(total_chars/len(governance_manual)-1)*100:.1f}% duplicate text)")

### Exercise 2: Markdown Header Aware Chunking
LangChain's `MarkdownHeaderTextSplitter` preserves the section hierarchy in metadata (e.g., `Header 1: ..., Header 2: ...`).

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = md_splitter.split_text(governance_manual)

print(f"Markdown Header Splits: {len(md_header_splits)}")
for split in md_header_splits[:2]:
    print(f"Metadata: {split.metadata}")
    print(f"Content: {split.page_content[:100]}...\n")

## Key Takeaways

| Strategy | Best Suited For | Critical Considerations |
|---|---|---|
| **Fixed-Size** | Baseline algorithmic experiments | Often breaks words and disrupts syntax |
| **Sentence-Aware** | NLP summarization & classification | Variance in sentence lengths |
| **Recursive Character** | General RAG documentation & manuals | Preserves paragraphs, sentences, and words |
| **Token-Based** | Strict context budgeting (e.g. GPT-4) | Disregards semantic paragraph flow |
| **Semantic Chunking** | Dynamic topic transition detection | Requires embedding model during chunking step |